# Homework: Agentic RAG

## Preparation

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
files[0]

RawRepositoryFile(filename='01-agentic-rag/lessons/01-intro.md', content='# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are 

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [29]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Q1. How many lesson pages

In [5]:
len(documents)

72

## Q2. Indexing and searching

In [6]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [30]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

search_results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

## Q3. Indexing and searching

In [8]:
from pathlib import Path
import sys

current = Path.cwd().resolve()

for directory in [current, *current.parents]:
    if (directory / "src" / "rag_helper.py").exists():
        sys.path.insert(0, str(directory))
        break
else:
    raise FileNotFoundError("Cannot find src/rag_helper.py")

from src.rag_helper import RAGBase
from dataclasses import dataclass
from typing import Any


@dataclass
class RAGResult:
    answer: str
    usage: Any
    search_results: list[dict]

class LessonsRAG(RAGBase):
    def search(self, query: str, num_results: int = 5):
        return self.index.search(
            query=query,
            num_results=num_results,
        )

    def build_context(self, search_results: list[dict]) -> str:
        contexts = []

        for doc in search_results:
            contexts.append(
                f"Filename: {doc['filename']}\n"
                f"Content:\n{doc['content']}"
            )

        return "\n\n---\n\n".join(contexts)

    def llm(self, prompt: str):
        input_messages = [
            {
                "role": "developer",
                "content": self.instructions,
            },
            {
                "role": "user",
                "content": prompt,
            },
        ]

        return self.llm_client.responses.create(
            model=self.model,
            input=input_messages,
        )

    def rag(self, query: str) -> RAGResult:
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)

        return RAGResult(
        answer=response.output_text,
        usage=response.usage,
        search_results=search_results,
    )

In [9]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_LLM_ENDPOINT_PROVIDER")
)

In [10]:
ZoomLLMLessonsAssistant = LessonsRAG(
    index=index,
    model="gemma-4-31b",
    llm_client=openai_client,
)

In [11]:
question = (
    "How does the agentic loop keep calling "
    "the model until it stops?"
)

result = ZoomLLMLessonsAssistant.rag(question)

print(result.answer)
print("Input tokens:", result.usage.input_tokens)
print(
    "Retrieved files:",
    [doc["filename"] for doc in result.search_results],
)

The agentic loop keeps calling the model by using a `while` loop and a tracking flag (such as `has_function_calls`). The process works as follows:

1.  **Model Request:** The code sends the current message history and available tools to the model.
2.  **Checking for Tool Calls:** The loop iterates through the model's response. If the response contains a `function_call`, the `has_function_calls` flag is set to `True`.
3.  **Execution:** When a function call is detected, the code executes the corresponding tool (e.g., a `search` function), and the tool's output is appended to the message history.
4.  **Iteration:** Because `has_function_calls` is `True`, the loop continues, sending the updated history (which now includes the tool results) back to the model.
5.  **Stopping Condition:** The loop only stops (breaks) when the model returns a response that contains no further function calls (i.e., `has_function_calls` remains `False`), indicating that the model has a final answer.
Input token

## Q4. Chunking

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [13]:
len(chunks)

295

## Q5. RAG with chunking

In [14]:
index_with_chunks = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index_with_chunks.fit(chunks)

In [15]:
ZoomLLMLessonsAssistant.index = index_with_chunks

In [16]:
result = ZoomLLMLessonsAssistant.rag(question)

In [17]:
print(result.answer)
print("Input tokens:", result.usage.input_tokens)
print(
    "Retrieved files:",
    [doc["filename"] for doc in result.search_results],
)

The agentic loop uses a `while True` loop that continues calling the model until the model returns a response that contains no function calls. 

The process works as follows:
1. The loop calls the model and checks the output for any `function_call` items.
2. If function calls are present, a flag (e.g., `has_function_calls`) is set to `True`, the tools are executed, and the results are added back to the message history for the next turn.
3. The loop stops (breaks) only when the model returns a final answer with no more tool calls, meaning `has_function_calls` remains `False` for that turn.
Input tokens: 2638
Retrieved files: ['01-agentic-rag/lessons/14-agentic-loop.md', '01-agentic-rag/lessons/14-agentic-loop.md', '01-agentic-rag/lessons/14-agentic-loop.md', '01-agentic-rag/lessons/15-frameworks.md', '01-agentic-rag/lessons/15-frameworks.md']


In [19]:
diff_input_tokens = int(7986 / 2638)
print(f"Input tokens reduced by a factor of {diff_input_tokens}x with chunking.")

Input tokens reduced by a factor of 3x with chunking.


## Q6. Turning it into an agent

In [24]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [25]:
def search(query: str) -> dict[str, str]:
    """
    Search the Lessons database for entries matching the given query.
    """
    return index_with_chunks.search(
        query,
        num_results=5,
    )

# Create tools
agent_tools = Tools()
agent_tools.add_tool(search)

In [26]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the Lessons database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [28]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

instructions = ("You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.")

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(
        model="gemma-4-31b",
        client=openai_client
    )
)

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


Feature,Plain RAG (Retrieval-Augmented Generation),Agentic Loop
Workflow,Linear/Fixed: Search $\rightarrow$ Augment Prompt $\rightarrow$ Generate Answer.,"Iterative/Dynamic: The LLM decides when to search, how many times, and what tools to use."
Decision Making,The system always performs a search before the LLM sees the query.,The LLM decides if a search is necessary and evaluates the results to decide the next step.
Error Recovery,"If the search returns bad results, the LLM will likely generate a wrong answer or say ""I don't know.""","The agent can recognize poor search results and ""self-correct"" by trying a different search query."
Complexity,"Simple, deterministic, and lower latency/cost.","More complex; can involve multiple LLM calls, increasing cost and latency."
Use Case,When the sequence of steps is known and the process is repeatable.,"When steps are unknown in advance or depend on dynamic, changing information."
